In [ ]:
using MLJ
using MLJFlux                  # interfaz para modelos Flux
using MLJScikitLearnInterface  # interfaz para SVM (scikit-learn wrapper)
using NearestNeighborModels    # interfaz para KNN
using Random
using LIBSVM
using Pkg
using MLJModelInterface
Pkg.precompile()


In [ ]:
using MLJ
using MLJFlux
using MLJScikitLearnInterface
using NearestNeighborModels
using Random
using LIBSVM
using Pkg
using MLJModelInterface
using MLJBase
using CategoricalArrays
using Statistics
using DataFrames
using CSV
using Tables
using Plots

import MLJBase: fit!, transform, machine, predict, predict_mode

In [ ]:
include("P2.jl")
include("PEARSON.jl")
include("SPEARMAN.jl")
include("ANOVA.jl")
include("KENDALL_TAU.jl")
include("RFE.jl")
include("MUTUALINFORMATION.jl")

# 1. Preparación de los datos

## 1.1. Carga y unificación de datos

In [ ]:
unifyDataset("./DatosPractica", "./dataset.csv") # Se leen todos los datos del árbol de carpetas y se guardan en un único csv

df = CSV.read("./dataset.csv", DataFrame) # Lectura de ese mismo csv

subjects, data, targets = separateDataframe(df)

println("Numero de variables: $(length(data[1,:]))")
println("Numero de instancias: $(length(data[:,1]))")
println("Número de sujetos: $(length(unique(subjects)))")
println("Número de clases de salida: $(length(unique(targets)))")


## 1.2. Análisis de valores ausentes

In [ ]:
missings_per_variable = sum(ismissing.(x) for x in eachcol(data))

# TODO: Transformar a heatmap o algo 
for col in 1:length(data[1,:])
    println("$(names(data)[col]): $(round((missings_per_variable[col] * 100)/(length(data[1,:])), sigdigits = 5))%")
end

missingPercentage = (sum(missings_per_variable)*100)/(length(data[1,:])*length(data[:,1]))
println("Porcentaje de valores faltantes: $(round(missingPercentage, sigdigits = 5))%")

## 1.3. Tratamiento de datos

In [ ]:
# Eliminar missings del dataset
# TODO: hacer media por individuo
replaceWithMean!(data,subjects)

missings = sum(sum(ismissing.(x) for x in eachcol(data)))
missingPercentage = (missings*100)/(length(data[1,:])*length(data[:,1]))
println("Porcentaje de valores faltantes: $(round(missingPercentage, sigdigits = 5))%")

# Convertir el dataset a Array para trabajar más fácil con él
data = Array{Float32}(data)

In [ ]:
targets = oneHotEncoding(targets)

## 1.4. Partición Holdout

In [ ]:
using Random

Random.seed!(104)

trainSubjectNumbers, testSubjectNumbers = holdOut(length(unique(subjects)), 0.1)

println("Test subjects: $testSubjectNumbers")

testIndices = findall(x-> x in testSubjectNumbers, subjects)

testData = data[testIndices,:]
testTargets = targets[testIndices,:]
testSubjects = subjects[testIndices]
println(unique(testSubjects))

println("Train subjects: $(sort(trainSubjectNumbers))")

trainIndices = findall(x-> x in trainSubjectNumbers, subjects)

trainData = data[trainIndices,:]
trainTargets = targets[trainIndices,:]
trainSubjects = subjects[trainIndices]
println(unique(trainSubjects))

## 1.5. Validación cruzada individual-wise

In [ ]:
Random.seed!(104)

foldTrainData, foldTrainTargets, foldValData, foldValTargets = individualWiseFoldCrossValidation(trainSubjects, trainData, trainTargets, 5)

## 1.6. Normalización

In [ ]:
using MLJModelInterface
using MLJBase
using Tables
using Statistics
import MLJModelInterface: fit, transform, input_scitype, target_scitype, output_scitype, predict

mutable struct MinMaxNormalizer <: Unsupervised
    mins::Vector{Float64}
    maxs::Vector{Float64}
end

MinMaxNormalizer() = MinMaxNormalizer(Float64[], Float64[])


function fit(model::MinMaxNormalizer, verbosity::Int, X)
    Xmat = Float64.(MLJBase.matrix(X))

    mins = mapslices(minimum, Xmat; dims=1) |> vec
    maxs = mapslices(maximum, Xmat; dims=1) |> vec

    fitresult = (mins, maxs)
    cache = nothing
    report = nothing

    return fitresult, cache, report
end

function transform(
    model::MinMaxNormalizer,
    fitresult,
    X
)
    mins, maxs = fitresult
    Xmat = MLJBase.matrix(X)

    Xscaled = (Xmat .- mins') ./ (maxs' .- mins')
    return MLJBase.table(Xscaled)
end

MLJModelInterface.input_scitype(::Type{MinMaxNormalizer}) = Table(Continuous)
MLJModelInterface.output_scitype(::Type{MinMaxNormalizer}) = Table(Continuous)


# 2. Modelos básicos y selección de atributos

In [ ]:

# PCA = @load PCA pkg=MultivariateStats add=true
# ICA = @load ICA pkg=MultivariateStats
# LDA = @load LDA pkg=MultivariateStats

# KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels
# SVC = @load ProbabilisticSVC pkg=LIBSVM verbosity=0
# NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux

# reducer=Dict(
#     "Sin Reducción" => nothing,
#     "PCA" => PCA(maxoutdim=50),
#     "ICA" => ICA(outdim=50, maxiter=500, tol=1, do_whiten=true),
#     "LDA" => LDA(outdim=50)
# )

# filter = Dict(
#     "Sin filtrado" => nothing,
#     "ANOVA" => ANOVAFilter(k=50),
#     "PEARSON" => PearsonFilter(k=50),
#     "SPEARMAN" => SpearmanFilter(k=50),
#     "KENDALLTAU" => KendallFilter(k=50),
#     "MI" => MutualInformation(k=50),
#     "RFE" => RFE(k=50)
# )

# models = Dict(
#     "KNN_1" => KNNClassifier(K=1),
#     "KNN_10" => KNNClassifier(K=10),
#     "KNN_20" => KNNClassifier(K=20),
#     "SVM_0.1" => SVC(cost=0.1),
#     "SVM_0.5" => SVC(cost=0.5),
#     "SVM_1" => SVC(cost=1.0),
#     "MLP_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
#     "MLP_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
#     "MLP_100-50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))) 
# )



In [ ]:
function evaluate_pipeline_cv_manual(filter_model, reducer_model, clf_model;
                                     foldTrainData, foldTrainTargets,
                                     foldValData, foldValTargets,
                                     n_folds=5, verbosity=0)
    
    fold_accuracies = Float64[]
    
    for fold in 1:n_folds
        try
            # Convertir datos
            X_train = MLJBase.table(foldTrainData[fold])
            y_train = categorical(findfirst.(x -> x==1, eachrow(foldTrainTargets[fold])))
            X_val = MLJBase.table(foldValData[fold])
            y_val = categorical(findfirst.(x -> x==1, eachrow(foldValTargets[fold])))
            
            X_train_transformed = X_train
            X_val_transformed = X_val
            
            # Normalización
            minmax = MinMaxNormalizer()
            minmax_mach = machine(minmax, X_train_transformed)
            fit!(minmax_mach, verbosity=verbosity)
            X_train_transformed = transform(minmax_mach, X_train_transformed)
            X_val_transformed = transform(minmax_mach, X_val_transformed)
            
            # Filtrado (si existe)
            if filter_model !== nothing
                filter_mach = machine(filter_model, X_train_transformed, y_train)
                fit!(filter_mach, verbosity=verbosity)
                X_train_transformed = transform(filter_mach, X_train_transformed)
                X_val_transformed = transform(filter_mach, X_val_transformed)
            end
            
            # Reducción (si existe)
            if reducer_model !== nothing
                if MLJModelInterface.is_supervised(reducer_model)
                    reducer_mach = machine(reducer_model, X_train_transformed, y_train)
                else
                    reducer_mach = machine(reducer_model, X_train_transformed)
                end
                fit!(reducer_mach, verbosity=verbosity)
                X_train_transformed = transform(reducer_mach, X_train_transformed)
                X_val_transformed = transform(reducer_mach, X_val_transformed)
            end
            
            # Clasificación
            clf_mach = machine(clf_model, X_train_transformed, y_train)
            fit!(clf_mach, verbosity=verbosity)
            y_pred = predict_mode(clf_mach, X_val_transformed)
            
            acc = mean(y_pred .== y_val)
            push!(fold_accuracies, acc)
            
        catch e
            if verbosity > 0
                println("Error en fold $fold: $e")
            end
            return nothing
        end
    end
    
    return mean(fold_accuracies), std(fold_accuracies)
end

In [ ]:
using MLJ
using MLJBase
using CategoricalArrays
using Statistics
using DataFrames
using CSV

# ============================================
# CARGAR MODELOS
# ============================================

println("Cargando modelos\n")

PCA = @load PCA pkg=MultivariateStats
ICA = @load ICA pkg=MultivariateStats
LDA = @load LDA pkg=MultivariateStats

KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels
SVC = @load ProbabilisticSVC pkg=LIBSVM 
NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux


filters = Dict(
    # "ANOVA_50" => ANOVAFilter(k=50),
    # "Pearson_50" => PearsonFilter(k=50),
    # "Spearman_50" => SpearmanFilter(k=50),
    # "Kendall_50" => KendallFilter(k=50),
    # "MI_50" => MutualInformation(k=50),
    "RFE_50" => RFE(k=50),
    # "None" => nothing  # Sin filtro
)

reducers = Dict(
    "PCA_5" => PCA(maxoutdim=10),
    "ICA_5" => ICA(outdim=10, maxiter=500, tol=1, do_whiten=true),
    "LDA_5" => LDA(),
    "None" => nothing  # Sin reducción
)

models = Dict(
    "KNN_1" => KNNClassifier(K=1),
    # "KNN_10" => KNNClassifier(K=10),
    # # "KNN_20" => KNNClassifier(K=20),
    # "SVM_0.1" => SVC(cost=0.1),
    # "SVM_0.5" => SVC(cost=0.5),
    # "SVM_1" => SVC(cost=1.0),
    # "MLP_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    # "MLP_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    # "MLP_100-50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50)))
)




# ============================================
# EVALUAR TODAS LAS COMBINACIONES
# ============================================

results = DataFrame(
    Filter=String[], 
    Reducer=String[], 
    Model=String[], 
    MeanAcc=Float64[],
    StdAcc=Float64[]
    )

total_combinations = length(filters) * length(reducers) * length(models)
current = 0

println("$total_combinations combinaciones con 5-Fold CV")
println("MinMaxNormalizer → Filter → Reducer → Classifier")
println("="^80)
println()

for(mname, mod) in models
    for (fname, filt) in filters
        for (rname, red) in reducers
# for (fname, filt) in filters
#     for (rname, red) in reducers
#         for (mname, mod) in models
            current += 1
            

            
            print("[$current/$total_combinations] $(rpad(fname, 12)) + $(rpad(rname, 10)) + $(rpad(mname, 14)) ")
            
            result = evaluate_pipeline_cv_manual(
                filt, red, mod;
                foldTrainData=foldTrainData,
                foldTrainTargets=foldTrainTargets,
                foldValData=foldValData,
                foldValTargets=foldValTargets
            )
            
            if result !== nothing
                mean_acc, std_acc = result
                push!(results, (fname, rname, mname, mean_acc, std_acc))
                println("$(round(mean_acc * 100, digits=2))% ± $(round(std_acc * 100, digits=2))%")
            end
            
        end
    end
end


sort!(results, :MeanAcc, rev=true)

println("\n" * "="^90)
println("TOP COMBINACIONES")
println("="^90)

for (i, row) in enumerate(eachrow(first(results, min(10, nrow(results)))))
    println("$(rpad(i, 3)). $(rpad(row.Filter, 14)) + $(rpad(row.Reducer, 12)) + $(rpad(row.Model, 14)) → $(round(row.MeanAcc * 100, digits=2))% ± $(round(row.StdAcc * 100, digits=2))%")
end



println("\n" * "="^90)
println("ESTADÍSTICAS GENERALES")
println("="^90)

println("Total de combinaciones evaluadas: $(nrow(results))")
if nrow(results) > 0
    println("Mejor accuracy: $(round(maximum(results.MeanAcc) * 100, digits=2))%")
    println("Peor accuracy: $(round(minimum(results.MeanAcc) * 100, digits=2))%")
    println("Accuracy promedio: $(round(mean(results.MeanAcc) * 100, digits=2))%")
    
    
    # Mejores por modelo
    println("\nMejor combinación por modelo:")
    for mname in unique(results.Model)
        best = first(sort(filter(row -> row.Model == mname, results), :MeanAcc, rev=true))
        println("  $(rpad(mname, 14)): $(round(best.MeanAcc * 100, digits=2))% ($(best.Filter) + $(best.Reducer))")
    end
end



In [ ]:
function train_test_models(clf_model;
                                     foldTrainData, foldTrainTargets,
                                     foldValData, foldValTargets,
                                     n_folds=5, verbosity=0)
    
    fold_accuracies = Float64[]
    
    for fold in 1:n_folds
        try
            # Convertir datos
            X_train = MLJBase.table(foldTrainData[fold])
            y_train = categorical(findfirst.(x -> x==1, eachrow(foldTrainTargets[fold])))
            X_val = MLJBase.table(foldValData[fold])
            y_val = categorical(findfirst.(x -> x==1, eachrow(foldValTargets[fold])))
            
            X_train_transformed = X_train
            X_val_transformed = X_val
            
            # Normalización
            minmax = MinMaxNormalizer()
            minmax_mach = machine(minmax, X_train_transformed)
            fit!(minmax_mach, verbosity=verbosity)
            X_train_transformed = transform(minmax_mach, X_train_transformed)
            X_val_transformed = transform(minmax_mach, X_val_transformed)
            
            # Clasificación
            clf_mach = machine(clf_model, X_train_transformed, y_train)
            fit!(clf_mach, verbosity=verbosity)
            y_pred = predict_mode(clf_mach, X_val_transformed)
            
            acc = mean(y_pred .== y_val)
            push!(fold_accuracies, acc)
            
        catch e
            if verbosity > 0
                println("Error en fold $fold: $e")
            end
            return nothing
        end
    end
    
    return mean(fold_accuracies), std(fold_accuracies)
end

In [ ]:
using MLJ
# using MLJEnsembles: EnsembleModel
# BaggingClassifier = @load EnsembleModel pkg=MLJEnsembles
KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels
AdaBoostClassifier = @load AdaBoostClassifier pkg=MLJScikitLearnInterface verbosity=0
AdaBoostStumpClassifier = @load AdaBoostStumpClassifier pkg=DecisionTree

DecisionTreeClassifier = @load DecisionTreeClassifier pkg=DecisionTree
EvoTreeClassifier = @load EvoTreeClassifier pkg=EvoTrees verbosity=0


EvoTreeClassifier(n_estimators = 50, learning_rate = .2)
evoTree100 = EvoTreeClassifier(n_estimators = 100, learning_rate = .2)
adaBoost = AdaBoostClassifier(estimator = DecisionTreeClassifier(), n_estimators = 5)
bagging10 = BaggingClassifier(estimator = KNNClassifier(K = 5),n_estimators = 10)
bagging50 = BaggingClassifier(estimator = KNNClassifier(K = 5),n_estimators = 50)

ensembles = Dict(
    "EvoTree50" => EvoTreeClassifier(n_estimators = 50, learning_rate = .2),
    "EvoTree100" => EvoTreeClassifier(n_estimators = 100, learning_rate = .2),
    # "AdaBoost" = AdaBoostClassifier(estimator = DecisionTreeClassifier(), n_estimators = 5),
    # "bagging50" => BaggingClassifier(estimator = KNNClassifier(K = 5),n_estimators = 50),
    # "Bagging100" => BaggingClassifier(estimator = KNNClassifier(K = 5),n_estimators = 100)
)
results = DataFrame(
    Model=String[], 
    MeanAcc=Float64[],
    StdAcc=Float64[]
    )
for (nmodel, ensemble) in ensembles
    result = evaluate_pipeline_cv_manual(
                nothing, nothing, ensemble;
                foldTrainData=foldTrainData,
                foldTrainTargets=foldTrainTargets,
                foldValData=foldValData,
                foldValTargets=foldValTargets
            )
    if result !== nothing
                mean_acc, std_acc = result
                push!(results, (nmodel, mean_acc, std_acc))
                println("$(round(mean_acc * 100, digits=2))% ± $(round(std_acc * 100, digits=2))%")
    end
end

In [ ]:
# EJEMPLO DE ENTRENAMIENTO
using CategoricalArrays
using MLJBase

AdaBoostClassifier = @load AdaBoostClassifier pkg=MLJScikitLearnInterface verbosity=0
DecisionTreeClassifier = @load DecisionTreeClassifier pkg=DecisionTree
KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels
adaBoost = AdaBoostClassifier(estimator = DecisionTreeClassifier(), n_estimators = 5)
knn = KNNClassifier(K=10)
minmax = MinMaxNormalizer()
mi = MutualInformation(k = 50)

values, _ ,_ = fit(minmax, 0, testData)
Xmat = transform(minmax, values, testData)

stats, _, _ = fit(mi, 0, Xmat, testTargets)
Xmat = transform(mi, stats, testData)

y_numeric = findall.(x -> x==1, eachrow(testTargets))
mach = machine(knn, Xmat, categorical.(y_numeric))
fit!(mach)
y_pred = predict_mode(mach, Xmat)
# acc = mean(y_pred .== y_test_cat)
# println("Test Accuracy: $(round(acc * 100, digits=2))%")

# 3. Modelos

In [ ]:
KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels
pipe = MinMaxNormalizer() |> KNNClassifier(K=2)

mach = machine(pipe, testData, testTargets)
fit!(mach)

In [ ]:
using Flux
NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux
pipe = MinMaxNormalizer() |> NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,)))
mach = machine(pipe, trainData, trainTargets)
fit!(mach)

In [ ]:
SVM = @load ProbabilisticSVC  pkg=LIBSVM  verbosity=0
model = SVM(C=1)
mach = machine(model, trainData, trainTargets)
fit!(mach)

In [ ]:
KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels
pipe = MinMaxNormalizer() |> KNNClassifier(K=10)

mach = machine(pipe, testData, testTargets)
fit!(mach)

predictions = predict(mach, testData) #ver lo que va mal con el predict en ello
accuracy_score = accuracy(predictions, testTargets)
println("Accuracy: $accuracy_score")

# 4 VISUALIZACIÓN

In [ ]:
using Plots
gr()

function plot_and_save(test_proj, test_targets, title_str)

    idxs = argmax(test_targets, dims=2)[:]
    labels = [idx[2] for idx in idxs]
    
    p = scatter(test_proj[:,1], test_proj[:,2], 
                group=labels, 
                title="$title_str - Test Set", 
                legend=:topright, 
                markersize=4, 
                alpha=0.8,
                size=(600,500))

    display(p)

    filename = lowercase(replace(title_str, " " => "_")) * "_test.png"
    fullpath = joinpath("visualizacion", filename)
    savefig(p, fullpath)
end

# 1. t-SNE
tsne_test = applyTSNE(testData; perplexity=30.0)
plot_and_save(tsne_test, testTargets, "t-SNE")

# 2. Isomap
isomap_test, test_comp = applyIsomap(testData; n_neighbors=20)
println("Isomap: $(length(test_comp))/$(size(testData,1)) muestras de test")
plot_and_save(isomap_test, testTargets[test_comp,:], "Isomap")

# 3. LLE
lle_test, test_comp = applyLLE(testData; n_neighbors=20)
println("LLE: $(length(test_comp))/$(size(testData,1)) muestras de test")
plot_and_save(lle_test, testTargets[test_comp,:], "LLE")